# Computer Vision Lab

## Landmarks — Where are the key points?

**Mission**

A landmark model returns meaningful coordinates, such as fingertips and hand joints.

In [ ]:
# Install MediaPipe because it supplies the pretrained hand-landmark model.
!pip -q install mediapipe


In [ ]:
# urlretrieve downloads the hand image and MediaPipe model file.
from urllib.request import urlretrieve
# display renders the original and annotated image below notebook cells.
from IPython.display import display
# Image opens the image; ImageDraw adds the landmark dots.
from PIL import Image, ImageDraw
# mp provides MediaPipe's image container.
import mediapipe as mp
# python provides BaseOptions for selecting the local model file.
from mediapipe.tasks import python
# vision provides the HandLandmarker task and its options.
from mediapipe.tasks.python import vision


### Predict

How many hands do you expect? Which finger points should be easiest for the model to locate?

In [ ]:
# IMAGE_URL is a public image with visible hands. Replace it only with consented images.
IMAGE_URL = "https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg"
# IMAGE_PATH is the local image filename after download.
IMAGE_PATH = "woman_hands.jpg"
# MODEL_PATH is the local filename for the pretrained hand-landmark model.
MODEL_PATH = "hand_landmarker.task"
# Download the input image to IMAGE_PATH.
urlretrieve(IMAGE_URL, IMAGE_PATH)
# Download the model bundle to MODEL_PATH.
urlretrieve("https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task", MODEL_PATH)


In [ ]:
# Open the input and convert it to RGB before showing it.
display(Image.open(IMAGE_PATH).convert("RGB"))


### Run the landmark model

The model returns coordinates; it does not identify the person.

In [ ]:
# BaseOptions tells MediaPipe where the downloaded model bundle is stored.
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
# num_hands=2 lets the model search for up to two hands in one image.
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
# Create one hand-landmark detector using the chosen model and options.
landmarker = vision.HandLandmarker.create_from_options(options)
# Convert the image file into MediaPipe's image format and run landmark detection.
result = landmarker.detect(mp.Image.create_from_file(IMAGE_PATH))
# len(...) counts the hands for which landmarks were returned.
print(f"Hands found: {len(result.hand_landmarks)}")


### Inspect the result

Which points would help recognise a pointing or pinching gesture?

In [ ]:
# POINT_RADIUS controls the radius, in pixels, of every displayed landmark dot.
POINT_RADIUS = 7
# canvas is a copy of the original image that will receive the yellow dots.
canvas = Image.open(IMAGE_PATH).convert("RGB")
# draw provides shape-drawing functions for the canvas image.
draw = ImageDraw.Draw(canvas)
# canvas.size returns image width first, then image height.
width, height = canvas.size
# Each hand contains normalized landmark coordinates from 0.0 to 1.0.
for hand in result.hand_landmarks:
    for point in hand:
        # Multiply normalized x and y by width and height to get image-pixel coordinates.
        x, y = int(point.x * width), int(point.y * height)
        # Draw a filled circle centered on this landmark coordinate.
        draw.ellipse((x - POINT_RADIUS, y - POINT_RADIUS, x + POINT_RADIUS, y + POINT_RADIUS), fill="yellow")
# Show the hand image with its predicted landmark points.
display(canvas)


### Change one thing

Try an image with an occluded hand. Which points become uncertain?

### Evidence

Why is a landmark system different from object detection? Use consented images only.